# Query Length Analysis

This notebook compares the lengths of original and synthetic queries
across the DutchNewsArticlesRetrieval, OpenTenderRetrieval, and
VABBRetrieval datasets.

For each query type, the analysis calculates descriptive statistics
(count, mean, median, standard deviation, minimum, and maximum).
Differences between original and synthetic query lengths are evaluated
using the Wilcoxon signed-rank test.

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

In [ ]:
news = pd.read_csv('news_full_augmented.csv')
tenders = pd.read_csv('tenders_full_augmented.csv')
abstracts = pd.read_csv('abstracts_full_augmented_clean.csv')

In [ ]:
datasets = {
    "News": news,
    "Tender": tenders,
    "VABB": abstracts
}

query_length_df = pd.concat(
    [
        df.assign(dataset=dataset)
        for dataset, df in datasets.items()
    ],
    ignore_index=True
)

print(query_length_df.shape)
print(query_length_df["dataset"].value_counts())

(3000, 8)
dataset
News      1000
Tender    1000
VABB      1000
Name: count, dtype: int64


In [ ]:
query_lengths = query_length_df.copy()

query_lengths["original_length"] = (
    query_lengths["original_query"]
    .astype(str)
    .str.split()
    .str.len()
)

query_lengths["synthetic_length"] = (
    query_lengths["synthetic_query"]
    .astype(str)
    .str.split()
    .str.len()
)

In [ ]:
length_summary = (
    query_lengths
    .groupby("dataset")[["original_length", "synthetic_length"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(length_summary)

original_length                             synthetic_length        \
                  count   mean median   std min max            count  mean   
dataset                                                                      
News               1000   6.77    6.0  2.50   1  15             1000  9.45   
Tender             1000   7.29    6.0  5.38   1  45             1000  8.60   
VABB               1000  10.05    9.0  5.59   1  39             1000  9.44   

                              
        median   std min max  
dataset                       
News       9.0  1.44   6  15  
Tender     9.0  1.30   6  13  
VABB       9.0  1.34   5  15

In [ ]:
overall_length = (
    query_lengths[["original_length", "synthetic_length"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)

display(overall_length)

,original_length,synthetic_length
mean,8.04,9.16
median,7.00,9.00
std,4.92,1.42
min,1.00,5.00
max,45.00,15.00


In [ ]:
from scipy.stats import wilcoxon

for dataset in query_lengths["dataset"].unique():
    subset = query_lengths[query_lengths["dataset"] == dataset]

    stat, p = wilcoxon(
        subset["original_length"],
        subset["synthetic_length"]
    )

    print(
        f"{dataset}: W = {stat:.0f}, p = {p:.4g}"
    )

News: W = 39134, p = 1.986e-103
Tender: W = 134166, p = 1.499e-26
VABB: W = 212966, p = 0.2293
